In [1]:
# ==========================================
# 1. IMPORTS
# ==========================================
import os
import ast
import wfdb
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [2]:
# ==========================================
# 2. PATHS
# ==========================================
BASE_PATH = "data/ptb-xl"
CSV_PATH = os.path.join(BASE_PATH, "ptbxl_database.csv")
IMAGE_PATH = os.path.join(BASE_PATH, "images")

In [3]:
# ==========================================
# 3. LOAD METADATA
# ==========================================
df = pd.read_csv(CSV_PATH)

In [ ]:
# ==========================================
# 4. BUILD DATASET (SIGNAL + IMAGE)
# ==========================================
signal_data = []
image_data = []
labels = []

for i, row in tqdm(df.iterrows(), total=len(df)):

    try:
        # ---------- SIGNAL ----------
        if use_high_res:   # your flag
            record_path = os.path.join(BASE_PATH, row['filename_hr'])
        else:
            record_path = os.path.join(BASE_PATH, row['filename_lr'])
        signal, _ = wfdb.rdsamp(record_path)

        # Normalize signal
        signal = (signal - np.mean(signal)) / (np.std(signal) + 1e-8)

        # ---------- IMAGE ----------
        img_path = os.path.join(IMAGE_PATH, f"{row['ecg_id']}.png")

        if not os.path.exists(img_path):
            continue

        img = load_img(img_path, target_size=(224,224))
        img = img_to_array(img) / 255.0

        # ---------- LABEL ----------
        scp_dict = ast.literal_eval(row['scp_codes'])

        if 'NORM' in scp_dict and scp_dict['NORM'] >= 50:
            label = 0   # normal
        else:
            label = 1   # abnormal

        # ---------- APPEND ----------
        signal_data.append(signal)
        image_data.append(img)
        labels.append(label)

    except:
        continue

# Convert to arrays
signal_data = np.array(signal_data)
image_data = np.array(image_data)
labels = np.array(labels)

print("Signal shape:", signal_data.shape)
print("Image shape:", image_data.shape)
print("Labels shape:", labels.shape)

100%|██████████| 21799/21799 [17:46<00:00, 20.44it/s]


Signal shape: (21799, 1000, 12)
Image shape: (21799, 224, 224, 3)
Labels shape: (21799,)


In [5]:
# ==========================================
# 5. TRAIN TEST SPLIT
# ==========================================
train_signal, test_signal, train_image, test_image, train_target, test_target = train_test_split(
    signal_data,
    image_data,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# Combine inputs as train_data / test_data
train_data = [train_signal, train_image]
test_data = [test_signal, test_image]


In [6]:
# ==========================================
# 6. SIGNAL MODEL (1D CNN)
# ==========================================
signal_input = Input(shape=(1000, 12))

x = Conv1D(32, 5, activation='relu')(signal_input)
x = MaxPooling1D(2)(x)

x = Conv1D(64, 5, activation='relu')(x)
x = MaxPooling1D(2)(x)

x = Conv1D(128, 5, activation='relu')(x)
x = MaxPooling1D(2)(x)

x = Flatten()(x)
x = Dense(128, activation='relu')(x)

signal_output = Dropout(0.5)(x)


In [7]:
# ==========================================
# 7. IMAGE MODEL (EfficientNet)
# ==========================================
image_input = Input(shape=(224,224,3))

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_tensor=image_input
)

base_model.trainable = False

y = base_model.output
y = Flatten()(y)
y = Dense(128, activation='relu')(y)

image_output = Dropout(0.5)(y)

In [8]:
# ==========================================
# 8. FUSION
# ==========================================
combined = concatenate([signal_output, image_output])

z = Dense(128, activation='relu')(combined)
z = Dropout(0.5)(z)

final_output = Dense(1, activation='sigmoid')(z)

In [9]:
# ==========================================
# 9. FINAL MODEL
# ==========================================
model = Model(inputs=[signal_input, image_input], outputs=final_output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 14,146,820 (53.97 MB)

 Trainable params: 10,097,249 (38.52 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [10]:
# ==========================================
# 10. TRAIN
# ==========================================
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.3)
]

history = model.fit(
    train_data,
    train_target,
    validation_data=(test_data, test_target),
    epochs=3,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/2
545/545 ━━━━━━━━━━━━━━━━━━━━ 554s 998ms/step - accuracy: 0.7610 - auc: 0.8409 - loss: 0.4983 - val_accuracy: 0.8422 - val_auc: 0.9259 - val_loss: 0.3457 - learning_rate: 3.0000e-04
Epoch 2/2
545/545 ━━━━━━━━━━━━━━━━━━━━ 525s 964ms/step - accuracy: 0.8463 - auc: 0.9207 - loss: 0.3517 - val_accuracy: 0.8546 - val_auc: 0.9353 - val_loss: 0.3328 - learning_rate: 3.0000e-04


In [11]:
# ==========================================
# 11. EVALUATION
# ==========================================
results = model.evaluate(test_data, test_target)
print("Test Loss, Accuracy, AUC:", results)

137/137 ━━━━━━━━━━━━━━━━━━━━ 102s 745ms/step - accuracy: 0.8546 - auc: 0.9353 - loss: 0.3328
Test Loss, Accuracy, AUC: [0.33276915550231934, 0.8545871376991272, 0.935342013835907]


In [12]:
# ==========================================
# 12. SAVE MODEL
# ==========================================
model.save("models/hybrid_ecg_model.h5")

print("✅ Hybrid ECG model trained and saved!")

✅ Hybrid ECG model trained and saved!
